In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
file_path = "/Tweets.csv"
df = pd.read_csv(file_path, encoding="ISO-8859-1")
df.head()


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [ ]:
df = df[['text', 'airline_sentiment']]
df.rename(columns={'airline_sentiment': 'sentiment'}, inplace=True)
df.head()


,text,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [ ]:
print(df.info())
print(df.isnull().sum())


enter the first Sentences: hello i am alone
Enter the second Sentences: i am happy when i am alone
Predictions for new tweets: [0 1]


In [ ]:
import re

def clean_tweet(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text.lower()

df['clean_text'] = df['text'].apply(clean_tweet)


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])

df['processed_text'] = df['clean_text'].apply(preprocess)
df[['text', 'processed_text']].head()


,text,processed_text
0,@VirginAmerica What @dhepburn said.,said
1,@VirginAmerica plus you've added commercials t...,plus youve added commercial experience tacky
2,@VirginAmerica I didn't today... Must mean I n...,didnt today must mean need take another trip
3,@VirginAmerica it's really aggressive to blast...,really aggressive blast obnoxious entertainmen...
4,@VirginAmerica and it's a really big bad thing...,really big bad thing


In [ ]:
sentiment_mapping = {'positive': 1, 'neutral': 0, 'negative': -1}
df['sentiment_label'] = df['sentiment'].map(sentiment_mapping)


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['processed_text']).toarray()
y = df['sentiment_label'].values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)


Training set size: (11712, 5000)
Test set size: (2928, 5000)


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.8005464480874317
              precision    recall  f1-score   support

          -1       0.82      0.94      0.88      1889
           0       0.68      0.49      0.57       580
           1       0.82      0.61      0.70       459

    accuracy                           0.80      2928
   macro avg       0.77      0.68      0.71      2928
weighted avg       0.79      0.80      0.79      2928



In [ ]:
while True:
    user_input = input("Enter a sentence (or type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break

    cleaned = preprocess(clean_tweet(user_input))
    vector = vectorizer.transform([cleaned]).toarray()
    pred = model.predict(vector)[0]

    mapping = {1:"Positive", 0:"Neutral", -1:"Negative"}
    print("Sentiment:", mapping[pred])


Enter a sentence (or type 'exit' to stop): hello
Sentiment: Neutral 😐
Enter a sentence (or type 'exit' to stop): i am alone
Sentiment: Negative ☹️
Enter a sentence (or type 'exit' to stop): hello i am alone
Sentiment: Negative ☹️
Enter a sentence (or type 'exit' to stop): hi i am mk
Sentiment: Neutral 😐
Enter a sentence (or type 'exit' to stop): i am passed
Sentiment: Neutral 😐
Enter a sentence (or type 'exit' to stop): i am excited
Sentiment: Positive 😊
Enter a sentence (or type 'exit' to stop): exit
